## Ablation analysis results: speech-only vs demographics-only vs combined model

In [3]:
import pandas as pd
from pathlib import Path
from datetime import datetime
now = lambda: datetime.now().strftime("%Y-%m-%d %H:%M")


RESULTS_DIR = Path("./ablation/MDC_results")
COHORT = "A"                        # column A only

DOMAINS = {                      # row group order
    "language": "Language",
    "memory": "Memory",
    "executive_function": "Executive function",
    "speed": "Speed",
}

VERSIONS = {                     # column order
    "speech_only": "Speech-only",
    "demographics_only": "Demographics-only",
    "kw19_withoutCohortB": "Combined",
}

METRICS = {                      # within-group row order
    "avg_speech_prediction_regression_r2_text": "$R^2$",
    "avg_speech_prediction_icc_text": "$ICC$",
    "avg_speech_prediction_mdc_speech_latent": "$MDC$",
    #"avg_speech_prediction_mdc_cognitive_text": "Cognitive $MDC$",
}

records = []
for domain, domain_label in DOMAINS.items():
    for version, version_label in VERSIONS.items():
        path = RESULTS_DIR / f"{domain}_{version}.csv"
        if not path.exists():
            print(f"[warn] missing file: {path}")
            continue

        s = pd.read_csv(path, index_col=0)[COHORT]

        # sanity check: file contents match the filename
        if s.get("target") != domain or s.get("version") != version:
            print(f"[warn] {path.name}: target/version = "
                  f"{s.get('target')}/{s.get('version')}")

        for key, metric_label in METRICS.items():
            if key not in s.index:
                print(f"[warn] {path.name}: no row '{key}'")

            value = s.get(key, pd.NA)
            if version == "demographics_only" and key == 'avg_speech_prediction_mdc_speech_latent':
                # violates between-person equals within-person $\lambda$ assumption of our model, so set to na here
                value = pd.NA

            records.append({
                "Composite Score": domain_label,
                "Metric": metric_label,
                "Model": version_label,
                "value": value,
            })

    # add cognitive reference, just takes the last version's value, but it's same for all versions
    records.append({
        "Composite Score": domain_label,
        "Metric": "$ICC$",
        "Model": "Cognitive assessment",
        "value": s.get("avg_speech_prediction_icc_cognitive_text", pd.NA),  #
    })
    records.append({
        "Composite Score": domain_label,
        "Metric": "$MDC$",
        "Model": "Cognitive assessment",
        "value": s.get("avg_speech_prediction_mdc_cognitive_text", pd.NA),
    })


table = (
    pd.DataFrame(records)
      .pivot(index=["Composite Score", "Metric"], columns="Model", values="value")
      .reindex(
          index=pd.MultiIndex.from_product(
              [list(DOMAINS.values()), list(METRICS.values())],
              names=["Composite Score", "Metric"],
          ),
          columns=list(VERSIONS.values()) + ["Cognitive assessment"],
      )
)
table.columns.name = None

display(table)
# print(table.to_latex(multirow=True, na_rep="--"))

Speech-only  Demographics-only  \
Composite Score    Metric                                        
Language           $R^2$   0.21 (0.15-0.27)  0.05 (-0.02-0.10)   
                   $ICC$   0.71 (0.66-0.76)   0.96 (0.94-0.97)   
                   $MDC$   2.06 (1.67-2.56)                NaN   
Memory             $R^2$   0.08 (0.02-0.12)   0.09 (0.02-0.14)   
                   $ICC$   0.73 (0.67-0.78)   0.97 (0.96-0.97)   
                   $MDC$   3.18 (2.43-4.55)                NaN   
Executive function $R^2$   0.12 (0.07-0.17)  0.05 (-0.01-0.10)   
                   $ICC$   0.64 (0.58-0.70)   0.93 (0.90-0.95)   
                   $MDC$   3.09 (2.44-4.05)                NaN   
Speed              $R^2$   0.08 (0.02-0.12)  0.01 (-0.03-0.05)   
                   $ICC$   0.65 (0.58-0.71)   0.89 (0.85-0.92)   
                   $MDC$   4.07 (3.09-5.70)                NaN   

                                   Combined Cognitive assessment  
Composite Score    Metric                                         
Language           $R^2$   0.25 (0.17-0.31)                  NaN  
                   $ICC$   0.80 (0.76-0.83)     0.69 (0.60-0.75)  
                   $MDC$   1.61 (1.33-1.98)     1.48 (1.29-1.67)  
Memory             $R^2$   0.14 (0.06-0.20)                  NaN  
                   $ICC$   0.88 (0.85-0.90)     0.68 (0.61-0.74)  
                   $MDC$   1.72 (1.35-2.36)     1.65 (1.48-1.81)  
Executive function $R^2$   0.18 (0.11-0.24)                  NaN  
                   $ICC$   0.76 (0.71-0.80)     0.71 (0.64-0.77)  
                   $MDC$   2.15 (1.73-2.73)     1.46 (1.30-1.64)  
Speed              $R^2$   0.09 (0.04-0.14)                  NaN  
                   $ICC$   0.73 (0.67-0.79)     0.76 (0.70-0.81)  
                   $MDC$   3.29 (2.46-4.50)     1.28 (1.13-1.42)

In [4]:
print(f"% Cohort {COHORT} speech vs demographics, {now()}")
print(table.to_latex(multirow=True, na_rep="--", label="s_tab:cohort_a_speech_vs_demographics",
caption="Comparison of regression performance $R^2$, test-retest reliability $ICC$, and minimal detectable change $MDC$ of the predictions of a model trained only on speech features (speech-only), only on demographic features (demographics-only), and our full model (combined) for Cohort A across the four composite cognitive scores. The $MDC$ of the demograhpics-only model is omitted, because it fundamentally violates the between-person equals within-person $\\lambda$ assumption of our model (see Sections~\\ref{sec:results_assumptions_tests} and \\ref{sec:pitt_analysis})").replace("\\cline{1-5}", "\\midrule").replace("\\cline{1-6}", "\\midrule"))

% Cohort A speech vs demographics, 2026-09-22 11:50
\begin{table}
\caption{Comparison of regression performance $R^2$, test-retest reliability $ICC$, and minimal detectable change $MDC$ of the predictions of a model trained only on speech features (speech-only), only on demographic features (demographics-only), and our full model (combined) for Cohort A across the four composite cognitive scores. The $MDC$ of the demograhpics-only model is omitted, because it fundamentally violates the between-person equals within-person $\lambda$ assumption of our model (see Sections~\ref{sec:results_assumptions_tests} and \ref{sec:pitt_analysis})}
\label{s_tab:cohort_a_speech_vs_demographics}
\begin{tabular}{llllll}
\toprule
 &  & Speech-only & Demographics-only & Combined & Cognitive assessment \\
Composite Score & Metric &  &  &  &  \\
\midrule
\multirow[t]{3}{*}{Language} & $R^2$ & 0.21 (0.15-0.27) & 0.05 (-0.02-0.10) & 0.25 (0.17-0.31) & -- \\
 & $ICC$ & 0.71 (0.66-0.76) & 0.96 (0.94-0.97) & 0.80